# Transformer の全体像

このノートブックでは、文章生成を実現する数理モデル「Transformer」の全体像を学びます。

## 目次
1. LLM と Transformer の関係
2. エンコーダとデコーダ
3. Transformer のアーキテクチャ
4. 翻訳の具体例で処理の流れを追う
5. 確率的予測モデルとしての出力生成
6. 自己回帰的生成（トークンを1つずつ出力する）
7. 関連性テーブルと Softmax
8. まとめ

## 1. LLM と Transformer の関係

ChatGPT のような AI は **LLM（Large Language Model：大規模言語モデル）** と呼ばれます。

OpenAI が開発する LLM には **GPT** という名前がついていますが、これは略語です：

| 略語 | 英語 | 意味 |
|------|------|------|
| **G** | Generative | 生成する（テキストや画像などを作り出す） |
| **P** | Pre-trained | 事前学習済み（大量のデータで予めパラメータを最適化してある） |
| **T** | Transformer | 本章で学ぶ数理モデルの名前 |

つまり **GPT = 事前学習済みの Transformer を使って文章を生成するモデル** です。

Transformer は 2017 年に Google が論文 "Attention Is All You Need" で発表した深層学習モデルで、
現在の最先端 AI のほとんどが Transformer を基礎としています。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# LLM と Transformer の関係を図示
fig, ax = plt.subplots(figsize=(10, 5))

# 各層を描画（外側から内側へ）
layers = [
    {"label": "ChatGPT（サービス）", "rect": (0.5, 0.3, 9, 4.2), "color": "#E8F5E9"},
    {"label": "GPT（LLM：大規模言語モデル）", "rect": (1.0, 0.7, 8, 3.4), "color": "#BBDEFB"},
    {"label": "Transformer（数理モデル）← 本章で学ぶ", "rect": (1.5, 1.1, 7, 2.6), "color": "#FFE0B2"},
    {"label": "Self-Attention（中核メカニズム）", "rect": (2.0, 1.5, 6, 1.8), "color": "#FFCDD2"},
]

for layer in layers:
    x, y, w, h = layer["rect"]
    rect = mpatches.FancyBboxPatch((x, y), w, h,
                                    boxstyle="round,pad=0.1",
                                    facecolor=layer["color"],
                                    edgecolor="gray", linewidth=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h - 0.3, layer["label"],
            ha='center', va='top', fontsize=11, fontweight='bold')

ax.set_xlim(0, 10)
ax.set_ylim(0, 5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('LLM と Transformer の関係', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 2. エンコーダとデコーダ

Transformer は大きく **エンコーダ（Encoder）** と **デコーダ（Decoder）** の2つのパートから構成されています。

| パート | 英語 | 役割 | たとえるなら |
|--------|------|------|-------------|
| **エンコーダ** | Encoder | 入力データを「意味の詰まった中間表現」に変換する | 英語を聞いて**理解する**脳の部分 |
| **デコーダ** | Decoder | 中間表現を受け取って、目的に応じた出力を生成する | 理解した内容を**日本語で話す**脳の部分 |

### 補足：LLM は Decoder のみ

原論文の Transformer は Encoder + Decoder の両方を使いますが、
GPT-3.5 などの LLM は **Decoder のみ** の構造を採用しています。

本章では理解のために Encoder-Decoder 両方を使う**英文和訳**を例に解説を進めます。

In [ ]:
# エンコーダとデコーダの役割を図示
fig, ax = plt.subplots(figsize=(12, 4))

# エンコーダ
encoder_box = mpatches.FancyBboxPatch((0.5, 1), 4, 2,
                                       boxstyle="round,pad=0.2",
                                       facecolor="#BBDEFB", edgecolor="#1976D2", linewidth=2)
ax.add_patch(encoder_box)
ax.text(2.5, 2.3, 'エンコーダ (Encoder)', ha='center', fontsize=13, fontweight='bold')
ax.text(2.5, 1.7, '入力を「理解」する', ha='center', fontsize=11, color='#1565C0')
ax.text(2.5, 1.2, '英語 → 意味の中間表現', ha='center', fontsize=10, color='gray')

# 矢印
ax.annotate('', xy=(5.8, 2), xytext=(4.7, 2),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(5.25, 2.3, '中間表現', ha='center', fontsize=9, color='gray')

# デコーダ
decoder_box = mpatches.FancyBboxPatch((6, 1), 4, 2,
                                       boxstyle="round,pad=0.2",
                                       facecolor="#FFE0B2", edgecolor="#F57C00", linewidth=2)
ax.add_patch(decoder_box)
ax.text(8, 2.3, 'デコーダ (Decoder)', ha='center', fontsize=13, fontweight='bold')
ax.text(8, 1.7, '理解した内容を「出力」する', ha='center', fontsize=11, color='#E65100')
ax.text(8, 1.2, '意味の中間表現 → 日本語', ha='center', fontsize=10, color='gray')

# 入力と出力
ax.text(2.5, 0.5, '"Mount Fuji looks\nbeautiful in spring."',
        ha='center', fontsize=10, style='italic', color='#1565C0')
ax.annotate('', xy=(2.5, 0.95), xytext=(2.5, 0.8),
            arrowprops=dict(arrowstyle='->', color='#1976D2', lw=1.5))

ax.text(8, 0.5, '「富士山は春に美しく見える。」',
        ha='center', fontsize=10, color='#E65100')
ax.annotate('', xy=(8, 0.95), xytext=(8, 0.8),
            arrowprops=dict(arrowstyle='<-', color='#F57C00', lw=1.5))

ax.set_xlim(-0.5, 11)
ax.set_ylim(0, 3.8)
ax.axis('off')
ax.set_title('エンコーダとデコーダの役割（英文和訳の例）', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Transformer のアーキテクチャ

Transformer の内部構造を見ていきましょう。

### エンコーダ側の構成（×N 回繰り返し）

| 順番 | 処理 | 何をしているか |
|------|------|----------------|
| 1 | Input Embedding | 単語を数値ベクトルに変換する |
| 2 | Positional Encoding | 単語の「順番」の情報を付加する |
| 3 | Multi-Head Attention | 各単語が他のどの単語と関係が深いかを計算する |
| 4 | Add & Norm | 残差接続と正規化（学習を安定させる技法） |
| 5 | Feed Forward | 全結合ニューラルネットワークで特徴を変換する |
| 6 | Add & Norm | もう一度、残差接続と正規化 |

### デコーダ側の構成（×N 回繰り返し）

| 順番 | 処理 | 何をしているか |
|------|------|----------------|
| 1 | Output Embedding | 出力側のトークンを数値ベクトルに変換する |
| 2 | Positional Encoding | 順番の情報を付加する |
| 3 | Masked Multi-Head Attention | 未来のトークンを見ないようにマスクして Attention を計算 |
| 4 | Add & Norm | 残差接続と正規化 |
| 5 | Multi-Head Attention | エンコーダの出力を参照して Attention を計算（Cross-Attention） |
| 6 | Add & Norm | 残差接続と正規化 |
| 7 | Feed Forward | 全結合ニューラルネットワーク |
| 8 | Add & Norm | 残差接続と正規化 |
| 9 | Linear → Softmax | 最終的に各単語の確率を出力する |

今はそれぞれの詳細がわからなくても大丈夫です。
まずは **全体の流れ** を掴むことが大事です。後のノートブックで一つずつ掘り下げていきます。

In [ ]:
# 図4.2: Transformer のアーキテクチャを横向きに可視化
fig, axes = plt.subplots(2, 1, figsize=(16, 8), gridspec_kw={'height_ratios': [1, 1]})

def draw_block(ax, x, y, w, h, label, color, fontsize=8):
    """ブロックを描画するヘルパー関数"""
    rect = mpatches.FancyBboxPatch((x, y), w, h,
                                    boxstyle="round,pad=0.05",
                                    facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', wrap=True)

def draw_arrow(ax, x1, y, x2, color='black'):
    """水平矢印を描画"""
    ax.annotate('', xy=(x2, y), xytext=(x1, y),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

# === エンコーダ (上段) ===
ax = axes[0]
ax.set_xlim(-1, 17)
ax.set_ylim(-0.5, 3)
ax.axis('off')
ax.set_title('エンコーダ (Encoder) ×N', fontsize=14, fontweight='bold', color='#1565C0')

# 入力テキスト
ax.text(-0.5, 1.25, '"Mount Fuji looks\nbeautiful in spring."',
        ha='center', va='center', fontsize=9, style='italic',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='gray'))
draw_arrow(ax, 0.7, 1.25, 1.3)

# エンコーダのブロック
blocks_enc = [
    (1.5, 'Input\nEmbedding', '#E3F2FD'),
    (3.8, 'Positional\nEncoding', '#E3F2FD'),
    (6.1, 'Multi-Head\nAttention', '#BBDEFB'),
    (8.4, 'Add &\nNorm', '#C8E6C9'),
    (10.7, 'Feed\nForward', '#BBDEFB'),
    (13.0, 'Add &\nNorm', '#C8E6C9'),
]

for x, label, color in blocks_enc:
    draw_block(ax, x, 0.5, 1.8, 1.5, label, color, fontsize=9)

# エンコーダ内の矢印
for i in range(len(blocks_enc) - 1):
    x1 = blocks_enc[i][0] + 1.8
    x2 = blocks_enc[i+1][0]
    draw_arrow(ax, x1 + 0.05, 1.25, x2 - 0.05)

# ×N の囲み
repeat_rect = mpatches.FancyBboxPatch((5.8, 0.2, ), 9.3, 2.0,
                                       boxstyle="round,pad=0.1",
                                       facecolor='none', edgecolor='#1565C0',
                                       linewidth=2, linestyle='dashed')
ax.add_patch(repeat_rect)
ax.text(15.3, 1.25, '×N', fontsize=14, fontweight='bold', color='#1565C0')

# 出力ラベル
ax.text(15.5, 0.3, 'エンコーダ\nの出力', ha='center', fontsize=9, color='#1565C0',
        bbox=dict(boxstyle='round', facecolor='#E3F2FD', edgecolor='#1565C0'))

# === デコーダ (下段) ===
ax = axes[1]
ax.set_xlim(-1, 17)
ax.set_ylim(-1, 3)
ax.axis('off')
ax.set_title('デコーダ (Decoder) ×N', fontsize=14, fontweight='bold', color='#E65100')

# [BOS] トークン入力
ax.text(-0.5, 1.25, '[BOS]',
        ha='center', va='center', fontsize=11, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='gray'))
draw_arrow(ax, 0.0, 1.25, 0.5)

# デコーダのブロック
blocks_dec = [
    (0.6, 'Output\nEmbedding', '#FFF3E0'),
    (2.4, 'Masked\nMulti-Head\nAttention', '#FFE0B2'),
    (4.2, 'Add &\nNorm', '#C8E6C9'),
    (6.0, 'Multi-Head\nAttention', '#FFE0B2'),
    (7.8, 'Add &\nNorm', '#C8E6C9'),
    (9.6, 'Feed\nForward', '#FFE0B2'),
    (11.4, 'Add &\nNorm', '#C8E6C9'),
    (13.2, 'Linear', '#FFCCBC'),
    (15.0, 'Softmax', '#FFCCBC'),
]

for x, label, color in blocks_dec:
    draw_block(ax, x, 0.5, 1.5, 1.5, label, color, fontsize=8)

# デコーダ内の矢印
for i in range(len(blocks_dec) - 1):
    x1 = blocks_dec[i][0] + 1.5
    x2 = blocks_dec[i+1][0]
    draw_arrow(ax, x1 + 0.02, 1.25, x2 - 0.02)

# エンコーダからの入力を示す矢印
ax.annotate('エンコーダの\n出力が入る', xy=(6.75, 2.05), xytext=(6.75, 2.7),
            fontsize=9, color='#1565C0', fontweight='bold',
            ha='center',
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))

# 出力
ax.text(16.2, 1.25, '[富士山]',
        ha='center', va='center', fontsize=11, fontweight='bold', color='#E65100',
        bbox=dict(boxstyle='round', facecolor='#FFF3E0', edgecolor='#E65100'))

# ×N の囲み
repeat_rect2 = mpatches.FancyBboxPatch((2.1, 0.2), 11.5, 2.0,
                                        boxstyle="round,pad=0.1",
                                        facecolor='none', edgecolor='#E65100',
                                        linewidth=2, linestyle='dashed')
ax.add_patch(repeat_rect2)

# PE の表記
ax.text(1.35, 2.3, '+PE', fontsize=8, color='gray', ha='center')

plt.tight_layout()
plt.show()

print("ポイント:")
print("  1. エンコーダは入力文（英語）を処理して中間表現を作る")
print("  2. デコーダはエンコーダの出力 + [BOS]トークンを受け取って翻訳を開始")
print("  3. デコーダの Multi-Head Attention にエンコーダの出力が入る（Cross-Attention）")
print("  4. 最終的に Softmax で次のトークンの確率を出力する")

## 4. 翻訳の具体例で処理の流れを追う

書籍の例文を使って、Transformer がどのように翻訳を行うか、ステップごとに見ていきましょう。

**入力**: `"Mount Fuji looks beautiful in spring."`

**出力（期待）**: `「富士山は春に美しく見える」`

### トークンとは？

**トークン** = テキストをデータ化した最小単位です。

本章では簡略化して「文章を個々の単語に分割したもの」として扱います。

```
"Mount Fuji looks beautiful in spring."
  ↓ トークン化
["Mount", "Fuji", "looks", "beautiful", "in", "spring"]
```

### BOS トークン

**BOS** = **B**egin **o**f **S**entence（文の開始）

エンコーダ側の処理が完了したことを示す「合図」のトークンです。
これがデコーダに入力されると、翻訳の出力が始まります。

In [ ]:
# トークン化のデモ
sentence = "Mount Fuji looks beautiful in spring."

# 簡易トークン化（単語分割）
# 実際の Transformer はもっと複雑なトークン化を行うが、ここでは単語単位で考える
tokens = sentence.replace('.', '').split()

print("=== トークン化 ===")
print(f"入力文: \"{sentence}\"")
print(f"トークン列: {tokens}")
print(f"トークン数: {len(tokens)}")
print()

# 各トークンにID（番号）を割り当てる
# 実際のモデルでは数万語の辞書を使うが、ここでは簡単な例で示す
vocab = {word: i+1 for i, word in enumerate(tokens)}
vocab['[BOS]'] = 0  # BOS トークンの ID は 0
vocab['[EOS]'] = len(tokens) + 1  # EOS（文の終了）トークン

print("=== 単語 → ID の対応（語彙辞書） ===")
for word, idx in vocab.items():
    print(f"  {word:15s} → {idx}")

print()
token_ids = [vocab[t] for t in tokens]
print(f"入力文のトークンID列: {token_ids}")

In [ ]:
# 図4.3: Transformer の翻訳処理フローを4ステップで可視化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

steps = [
    {
        "title": "ステップ1: 英文をエンコーダに入力",
        "desc": 'エンコーダが英文の各トークンを\n処理して「意味の表現」を作る',
        "encoder_color": "#BBDEFB",
        "decoder_color": "#EEEEEE",
        "input_text": '[Mount] [Fuji] [looks]\n[beautiful] [in] [spring]',
        "output_text": "",
        "arrow_enc_dec": False,
        "bos": False,
    },
    {
        "title": "ステップ2: エンコーダの出力をデコーダへ",
        "desc": 'エンコーダの出力（中間表現）が\nデコーダの Multi-Head Attention に渡される',
        "encoder_color": "#BBDEFB",
        "decoder_color": "#EEEEEE",
        "input_text": '[Mount] [Fuji] [looks]\n[beautiful] [in] [spring]',
        "output_text": "",
        "arrow_enc_dec": True,
        "bos": False,
    },
    {
        "title": "ステップ3: [BOS]トークンをデコーダに入力",
        "desc": '[BOS]（文の開始）がデコーダに入力され\n翻訳の出力が始まる',
        "encoder_color": "#BBDEFB",
        "decoder_color": "#FFE0B2",
        "input_text": '[Mount] [Fuji] [looks]\n[beautiful] [in] [spring]',
        "output_text": "",
        "arrow_enc_dec": True,
        "bos": True,
    },
    {
        "title": "ステップ4: 最初のトークンが出力される",
        "desc": 'デコーダが処理を行い\n最初のトークン [富士山] を出力',
        "encoder_color": "#BBDEFB",
        "decoder_color": "#FFE0B2",
        "input_text": '[Mount] [Fuji] [looks]\n[beautiful] [in] [spring]',
        "output_text": "[富士山]",
        "arrow_enc_dec": True,
        "bos": True,
    },
]

for idx, (ax, step) in enumerate(zip(axes.flat, steps)):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 6)
    ax.axis('off')
    ax.set_title(step['title'], fontsize=11, fontweight='bold')

    # エンコーダ
    enc = mpatches.FancyBboxPatch((0.5, 3), 3.5, 2.2,
                                   boxstyle="round,pad=0.2",
                                   facecolor=step['encoder_color'],
                                   edgecolor='#1565C0', linewidth=2)
    ax.add_patch(enc)
    ax.text(2.25, 4.6, 'エンコーダ', ha='center', fontsize=10, fontweight='bold')

    # 入力テキスト
    ax.text(2.25, 3.7, step['input_text'], ha='center', fontsize=8, family='monospace')

    # デコーダ
    dec = mpatches.FancyBboxPatch((5.5, 3), 3.5, 2.2,
                                   boxstyle="round,pad=0.2",
                                   facecolor=step['decoder_color'],
                                   edgecolor='#E65100', linewidth=2)
    ax.add_patch(dec)
    ax.text(7.25, 4.6, 'デコーダ', ha='center', fontsize=10, fontweight='bold')

    # 説明
    ax.text(5, 1.0, step['desc'], ha='center', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='gray', alpha=0.8))

    # エンコーダ→デコーダの矢印
    if step['arrow_enc_dec']:
        ax.annotate('', xy=(5.5, 4.1), xytext=(4.0, 4.1),
                    arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))

    # BOS トークン
    if step['bos']:
        ax.text(7.25, 2.5, '[BOS]', ha='center', fontsize=10, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray'))
        ax.annotate('', xy=(7.25, 2.95), xytext=(7.25, 2.75),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

    # 出力トークン
    if step['output_text']:
        ax.text(9.3, 4.1, step['output_text'], ha='center', fontsize=12,
                fontweight='bold', color='#E65100',
                bbox=dict(boxstyle='round', facecolor='#FFF3E0', edgecolor='#E65100', linewidth=2))

plt.suptitle('図4.3: Transformer による英文和訳の処理ステップ', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. 確率的予測モデルとしての出力生成

Transformer は **確率的予測モデル** です。

これはどういう意味でしょうか？

デコーダの最終段にある **Softmax** は、語彙辞書の全単語に対して「次に来る確率」を計算します。
そして、最も確率の高い単語が出力として選ばれます。

```
入力: [BOS] + エンコーダの出力
  ↓ デコーダで処理
  ↓ Softmax で確率計算
出力: {"富士山": 0.85, "山": 0.08, "春": 0.03, "東京": 0.01, ...}
  ↓ 最も確率が高いものを選択
結果: "富士山"
```

つまり Transformer は「正解を知っている」のではなく、**「次に来る可能性が最も高い単語」を予測している** のです。

In [ ]:
# Softmax による確率分布の例を可視化
# デコーダが「次のトークン」を予測する様子

# 語彙辞書の一部（日本語側）
japanese_vocab = ['富士山', '山', '春', '東京', '美しい', '見える', 'は', 'に', 'の', 'その他']

# デコーダが出力する各単語の確率（Softmax の出力）
# 最初のトークン予測: 「富士山」が最も確率が高い
probabilities = [0.82, 0.06, 0.03, 0.02, 0.02, 0.01, 0.01, 0.01, 0.01, 0.01]

fig, ax = plt.subplots(figsize=(10, 5))

# 確率の高い順にソート済み
colors = ['#E65100'] + ['#FFB74D'] * 2 + ['#FFE0B2'] * 7  # 最大確率を強調
bars = ax.barh(range(len(japanese_vocab)), probabilities, color=colors, edgecolor='gray')

# 確率値をバーの横に表示
for i, (prob, bar) in enumerate(zip(probabilities, bars)):
    ax.text(prob + 0.01, i, f'{prob:.0%}', va='center', fontsize=11, fontweight='bold')

ax.set_yticks(range(len(japanese_vocab)))
ax.set_yticklabels(japanese_vocab, fontsize=12)
ax.set_xlabel('確率（Softmax の出力）', fontsize=12)
ax.set_title('デコーダの出力: 最初のトークンの予測確率\n入力 = [BOS] + エンコーダの出力', fontsize=13, fontweight='bold')
ax.set_xlim(0, 1.0)
ax.invert_yaxis()

# 最大確率の単語を強調
ax.annotate('← これが選ばれる！', xy=(0.85, 0), fontsize=12, color='#E65100',
            fontweight='bold', va='center')

plt.tight_layout()
plt.show()

print("ポイント:")
print("  Transformer は『正解を知っている』のではなく、")
print("  『次に来る可能性が最も高い単語を予測する』確率的モデルです。")
print(f"  この例では『富士山』が {probabilities[0]:.0%} の確率で選ばれました。")

In [ ]:
# Softmax 関数の仕組みを実際に計算してみる

def softmax(x):
    """ソフトマックス関数: 任意の数値列を確率分布に変換する"""
    # オーバーフロー防止のため最大値を引く
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

# デコーダの最終層が出力する「生のスコア」（ロジットと呼ぶ）
# これは Softmax に通す前の値で、まだ確率ではない
logits = np.array([4.5, 2.1, 1.3, 0.8, 0.7, 0.2, 0.1, 0.0, -0.1, -0.5])

print("=== Softmax の計算過程 ===")
print()
print("Step 1: デコーダが出力する生のスコア（ロジット）")
for word, logit in zip(japanese_vocab, logits):
    print(f"  {word:6s}: {logit:5.1f}")

print()
print("Step 2: 各スコアの指数（e^x）を計算")
exp_values = np.exp(logits - np.max(logits))  # オーバーフロー防止
for word, exp_val in zip(japanese_vocab, exp_values):
    print(f"  e^({word:6s}のスコア) = {exp_val:.4f}")

print(f"\n  合計 = {np.sum(exp_values):.4f}")

print()
print("Step 3: 合計で割って確率にする（全部足すと1.0になる）")
probs = softmax(logits)
for word, prob in zip(japanese_vocab, probs):
    print(f"  {word:6s}: {prob:.4f} ({prob:.1%})")

print(f"\n  確率の合計 = {np.sum(probs):.4f}（= 1.0）")
print(f"\n→ 最も確率が高い『{japanese_vocab[np.argmax(probs)]}』が選ばれる！")

## 6. まとめ

| ポイント | 内容 |
|----------|------|
| **Transformer** | 2017年 Google 発表の深層学習モデル。現在の LLM の基盤 |
| **GPT** | Generative Pre-trained Transformer の略 |
| **エンコーダ** | 入力（英語）を「意味の中間表現」に変換する部分 |
| **デコーダ** | 中間表現を受け取って出力（日本語）を生成する部分 |
| **トークン** | テキストをデータ化した最小単位（ここでは単語） |
| **BOS** | Begin of Sentence。デコーダに翻訳開始を指示する合図 |
| **Softmax** | 生のスコアを確率分布に変換し、最も確率の高い単語を選ぶ |
| **確率的予測** | 正解を「知っている」のではなく、確率的に「予測」している |

### 重要な全体像

```
英語の文 → [エンコーダ] → 中間表現 → [デコーダ] → Softmax → 日本語のトークン
```

## 次のステップ

次のノートブックでは、Transformer の最初のステップである
**「トークンと単語埋め込み（Embedding）」** について詳しく学びます。

- 単語をどうやって数値（ベクトル）に変換するのか？
- なぜベクトルにする必要があるのか？

を掘り下げていきます。